# Visualizing the training data and augmentations

This notebook shows exactly what the encoder will receive during multi-crop
WS-DINO training:

1. **Dataset facts**: frame interval, frame_gap, t1 window, z window, crop sizes.
2. **Raw biology**: full field-of-view mean projections across the t1 window.
3. **One sample**: globals_t1, globals_t2 and locals as the model receives them.
4. **Multi-crop layout**: globals + locals overlaid on the t1 frame so you can
   eyeball whether locals are nested correctly inside their parent global.
5. **Intensity gate**: distribution of random crop means vs the gate threshold,
   plus example accepted/rejected crops — for sanity-checking the gate.
6. **Depth montage**: the z slab inside one global.
7. **Augmentation grid**: 16 independent augmentations of the same crop.
8. **Per-augmentation isolation**: effect of each augmentation on its own.
9. **One real batch**: what the DataLoader actually hands to the model.

Each cell is independent. Re-run cells after editing `cfg` to see the effect.

In [ ]:
# setup: build cfg, pick zarr (or use multi-zarr from yaml), build dataset, define helpers.
import sys, os, time, copy
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from configs.config import Config
from data.dataset import ZarrCellDataset, MultiZarrDataset, worker_init_fn
from data.picker import pick_zarr_path
from data.augmentations import build_augmentations

# load defaults; swap for `Config.from_yaml('../configs/train.yaml')` to use the yaml.
cfg = Config.from_yaml("../configs/train.yaml")
# fresh seed per notebook run so each session sees different crops.
cfg.seed = time.time_ns() % (2**31 - 1)

# anchor any relative paths in the yaml to the project root so the notebook
# works regardless of where jupyter was launched from.
_project_root = os.path.abspath('..')
if not os.path.isabs(cfg.training_data_index_path):
    cfg.training_data_index_path = os.path.join(_project_root, cfg.training_data_index_path)

# build the right dataset shape.
if cfg.zarr_paths:
    ds = MultiZarrDataset(cfg)
    primary = ds.children[0]  # used for single-crop visualizations below
    is_multi = True
else:
    if cfg.zarr_path is None:
        cfg.zarr_path = pick_zarr_path()
    ds = ZarrCellDataset(cfg)
    primary = ds
    is_multi = False

sample_iter = iter(ds)

print('is_multi             :', is_multi)
print('seed                 :', cfg.seed)
print('n_global_crops       :', cfg.n_global_crops)
print('n_local_crops        :', cfg.n_local_crops)
print('global_crop_size     :', cfg.global_crop_size)
print('local_crop_size      :', cfg.local_crop_size)
print('z_thickness          :', cfg.z_thickness)
print('intensity_min_frac   :', cfg.intensity_min_frac)
print('target_gap_minutes   :', cfg.target_gap_minutes)
print('temporal_direction   :', cfg.temporal_direction)
print()
if is_multi:
    print(f'mixed across {len(ds.children)} datasets:')
    for i, c in enumerate(ds.children):
        print(f'  [{i}] {c.cfg.zarr_path}')
        print(f'      shape={tuple(c.arr.shape)} interval_min={c.interval_minutes} frame_gap={c.frame_gap}')
else:
    print('zarr_path            :', primary.cfg.zarr_path)
    print('arr.shape (T,C,Z,Y,X):', tuple(primary.arr.shape))
    print('interval_minutes     :', primary.interval_minutes)
    print('frame_gap (frames)   :', primary.frame_gap)
    print(f'wall-clock gap       : {primary.frame_gap * primary.interval_minutes:.1f} min')
    print('t1 window            :', (primary.t1_min, primary.t1_max))
    print('z window             :', (primary.z_min, primary.z_max))


# ----- visualization helpers -----

def mean_projection(volume):
    """average across z; gives a clean 2d view of a (z, h, w) volume."""
    return volume.mean(axis=0)


def mip(volume):
    """max projection across z; bright structures stand out."""
    return volume.max(axis=0)


def z_montage(volume, ncols=5):
    """tile z slices side by side into one image."""
    z, h, w = volume.shape
    nrows = (z + ncols - 1) // ncols
    out = np.zeros((nrows * h, ncols * w), dtype=volume.dtype)
    for i in range(z):
        r, c = divmod(i, ncols)
        out[r*h:(r+1)*h, c*w:(c+1)*w] = volume[i]
    return out


def percentile_limits(values, low=1.0, high=99.0):
    """display limits robust to a few bright outliers."""
    lo, hi = np.percentile(np.asarray(values), [low, high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.min(values)), float(np.max(values))
    if hi <= lo:
        hi = lo + 1e-6
    return float(lo), float(hi)


def show_gray(ax, image, title, limits):
    ax.imshow(image, cmap='gray', vmin=limits[0], vmax=limits[1])
    ax.set_title(title, fontsize=10)
    ax.axis('off')


def raw_frame_mean(child, t, z0, z1):
    """full-frame mean projection in normalized [0, 1] intensity (pre-augmentation)."""
    vol = child.arr[int(t), child.cfg.channel, int(z0):int(z1), :, :].compute()
    vol = np.asarray(vol, dtype=np.float32)
    return (np.clip(vol, 0, child.cfg.max_intensity) / child.cfg.max_intensity).mean(axis=0)


def load_crop_tensor(child, t, z, y, x, size):
    """raw->normalized [1, z, h, w] tensor matching what _sample_pair produces pre-augment."""
    raw = child._load_crop_raw(int(t), int(z), int(y), int(x), int(size), int(size))
    return child._to_tensor(raw)


## 1. Raw biology: the field of view over time

Full-frame mean projections at evenly-spaced timepoints, in normalized `[0, 1]` intensity (clip → divide by `max_intensity`). No augmentations, no cropping — this is what's *inside* the zarr.

In [ ]:
n_panels = 5
ts = np.linspace(primary.t1_min, primary.t1_max - 1, n_panels).astype(int)
z0 = primary.z_min
z1 = z0 + cfg.z_thickness

frames = [raw_frame_mean(primary, t, z0, z1) for t in ts]
lims = percentile_limits(np.concatenate([f.ravel() for f in frames]), 0.5, 99.5)

fig, axes = plt.subplots(1, n_panels, figsize=(3 * n_panels, 3.2))
for ax, f, t in zip(axes, frames, ts):
    mins = t * primary.interval_minutes
    show_gray(ax, f, f't={int(t)} ({mins:.0f} min)', lims)
plt.suptitle(f'full-frame mean z=[{z0},{z1}) across the t1 window', y=1.02)
plt.tight_layout(); plt.show()


## 2. One sample: globals_t1, globals_t2 and locals

Each sample from the dataset is a multi-crop bundle:

- `globals_t1`: `n_global_crops` larger views at frame `t1`
- `globals_t2`: same crops shifted to `t2 = t1 ± frame_gap`
- `locals`: `n_local_crops` smaller views at `t1`, each spatially nested inside one of the globals
- `weak_label`: a `(dataset_idx, frame_gap)` long tensor, the WS-DINO label the loss consumes

Each tensor is post-augmentation, in normalized units (per-crop z-scored when `aug_percrop_norm` is on).

In [ ]:
sample = next(sample_iter)
meta = sample['meta']
weak_label = sample['weak_label']

# resolve the child this sample came from so per-frame helpers work in the multi-zarr case.
src = ds.children[int(meta['dataset_idx'])] if is_multi else primary

g_t1 = sample['globals_t1'].numpy()  # (n_global, 1, z, h, w)
g_t2 = sample['globals_t2'].numpy()
has_locals = 'locals' in sample
loc = sample['locals'].numpy() if has_locals else None

print('sample keys          :', list(sample.keys()))
print('globals_t1 shape     :', g_t1.shape)
print('globals_t2 shape     :', g_t2.shape)
if has_locals:
    print('locals shape         :', loc.shape)
print('weak_label           :', weak_label.tolist(), '(dataset_idx, frame_gap)')
print('t1                   :', int(meta['t1']))
print('t2                   :', int(meta['t2']))
print('frame_gap            :', int(meta['frame_gap']))
print('z slab               :', (int(meta['z']), int(meta['z_end'])))
print('global offsets (y,x) :', list(zip(meta['global_y'].tolist(), meta['global_x'].tolist())))
if has_locals:
    print('local parents        :', meta['local_parent'].tolist())
    print('local offsets (y,x)  :', list(zip(meta['local_y'].tolist(), meta['local_x'].tolist())))

# globals: 2 rows (t1, t2) x n_global_crops columns.
n_g = g_t1.shape[0]
pool = np.concatenate([g_t1.ravel(), g_t2.ravel()])
glims = percentile_limits(pool, 1.0, 99.0)
fig, axes = plt.subplots(2, n_g, figsize=(3 * n_g, 6.2), squeeze=False)
for i in range(n_g):
    show_gray(axes[0, i], mean_projection(g_t1[i, 0]), f'global {i}  t1={int(meta["t1"])}', glims)
    show_gray(axes[1, i], mean_projection(g_t2[i, 0]), f'global {i}  t2={int(meta["t2"])}', glims)
plt.suptitle(f'globals (gap={int(meta["frame_gap"])} frames, dataset_idx={int(meta["dataset_idx"])})', y=1.02)
plt.tight_layout(); plt.show()

# locals: one row of n_local_crops tiles, titled with parent global index.
if has_locals:
    n_l = loc.shape[0]
    llims = percentile_limits(loc, 1.0, 99.0)
    ncols = min(n_l, 8)
    nrows = (n_l + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.4 * nrows), squeeze=False)
    parents = meta['local_parent'].tolist()
    for i in range(nrows * ncols):
        ax = axes[i // ncols, i % ncols]
        if i < n_l:
            show_gray(ax, mean_projection(loc[i, 0]), f'local {i}  parent={parents[i]}', llims)
        else:
            ax.axis('off')
    plt.suptitle(f'locals (n_local_crops={n_l}, local_crop_size={cfg.local_crop_size})', y=1.02)
    plt.tight_layout(); plt.show()
else:
    print('n_local_crops=0; skipping locals panel.')


## 3. Multi-crop layout: are the locals actually nested?

Globals are outlined on the full t1 frame in distinct colors. Each local is drawn in the same color as its parent global. If the geometry is right, every local rectangle sits fully inside the global rectangle that shares its color.

In [ ]:
frame = raw_frame_mean(src, int(meta['t1']), int(meta['z']), int(meta['z_end']))
flims = percentile_limits(frame, 0.5, 99.0)

g_size = cfg.global_crop_size
l_size = cfg.local_crop_size
colors = plt.cm.tab10(np.linspace(0, 1, max(n_g, 10)))

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(frame, cmap='gray', vmin=flims[0], vmax=flims[1])

gys = meta['global_y'].tolist()
gxs = meta['global_x'].tolist()
for i in range(n_g):
    rect = patches.Rectangle((gxs[i], gys[i]), g_size, g_size,
                             linewidth=2.5, edgecolor=colors[i], facecolor='none',
                             label=f'global {i}')
    ax.add_patch(rect)

if has_locals:
    parents = meta['local_parent'].tolist()
    lys = meta['local_y'].tolist()
    lxs = meta['local_x'].tolist()
    for j in range(len(parents)):
        p = parents[j]
        rect = patches.Rectangle((lxs[j], lys[j]), l_size, l_size,
                                 linewidth=1.0, edgecolor=colors[p], facecolor='none',
                                 linestyle='--')
        ax.add_patch(rect)

ax.legend(loc='upper right', fontsize=9)
ax.set_title(f'multi-crop layout at t={int(meta["t1"])} (z=[{int(meta["z"])},{int(meta["z_end"])}))')
ax.axis('off')
plt.tight_layout(); plt.show()


## 4. Intensity gate: are we actually rejecting background?

`_sample_pair` accepts a global crop only when its raw mean (over the z slab)
is at least `intensity_min_frac * frame_mean(t1)`. Two sanity checks:

- Histogram of crop means at many random `(y, x)` offsets, with the threshold
  drawn in. The gate should bisect background-only crops on the left from
  cell-containing crops on the right.
- A few example crops below and above the threshold.

If the histogram looks unimodal at the gate, the threshold is too aggressive
(or too permissive) for this dataset and `intensity_min_frac` needs a tweak.

In [ ]:
n_probe = 400
rng_probe = np.random.default_rng(cfg.seed + 1)
t_probe = int(meta['t1'])
z_probe = int(meta['z'])

# raw frame mean over the configured z window, in raw units (matches what the gate compares).
def raw_zslab_mean(child, t, z):
    sub = child.arr[int(t), child.cfg.channel, child.z_min:child.z_max, :, :].compute()
    return float(np.asarray(sub, dtype=np.float32).mean())

frame_mean_raw = raw_zslab_mean(src, t_probe, z_probe)
threshold_raw = cfg.intensity_min_frac * frame_mean_raw
print(f'frame_mean (raw)     : {frame_mean_raw:.2f}')
print(f'threshold (raw)      : {threshold_raw:.2f}  (intensity_min_frac={cfg.intensity_min_frac})')

# sample random crop means at random (y, x) offsets.
crop_means = []
crop_samples = []  # keep raw arrays for the example panel below
for _ in range(n_probe):
    y = int(rng_probe.integers(0, src.Y - g_size + 1))
    x = int(rng_probe.integers(0, src.X - g_size + 1))
    raw = src._load_crop_raw(t_probe, z_probe, y, x, g_size, g_size)
    crop_means.append(float(raw.mean()))
    crop_samples.append((y, x, raw))

crop_means = np.asarray(crop_means)
accept_frac = float((crop_means >= threshold_raw).mean())
print(f'random crop accept % : {100 * accept_frac:.1f}% of {n_probe} probes')

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(crop_means, bins=40, color='steelblue', alpha=0.85)
ax.axvline(threshold_raw, color='crimson', linewidth=2, label=f'threshold = {threshold_raw:.0f}')
ax.axvline(frame_mean_raw, color='black', linestyle='--', linewidth=1.2,
           label=f'frame mean = {frame_mean_raw:.0f}')
ax.set_xlabel('crop mean (raw units)')
ax.set_ylabel('count')
ax.set_title(f'random crop means at t={t_probe}, z={z_probe} (n={n_probe})')
ax.legend()
plt.tight_layout(); plt.show()

# example crops: 3 below threshold, 3 above. mean projection in normalized [0,1] for display.
below = sorted([s for s in crop_samples if s[2].mean() <  threshold_raw], key=lambda s: s[2].mean())[:3]
above = sorted([s for s in crop_samples if s[2].mean() >= threshold_raw], key=lambda s: -s[2].mean())[:3]
examples = below + above
if examples:
    norm_imgs = [(np.clip(r, 0, cfg.max_intensity) / cfg.max_intensity).mean(axis=0) for _, _, r in examples]
    lims = percentile_limits(np.concatenate([im.ravel() for im in norm_imgs]), 1.0, 99.0)
    fig, axes = plt.subplots(1, len(examples), figsize=(2.6 * len(examples), 3.0))
    if len(examples) == 1:
        axes = [axes]
    for ax, (y, x, raw), img in zip(axes, examples, norm_imgs):
        verdict = 'REJECT' if raw.mean() < threshold_raw else 'accept'
        show_gray(ax, img, f'{verdict}\nmean={raw.mean():.0f}\n(y,x)=({y},{x})', lims)
    plt.suptitle('example crops below/above the intensity gate', y=1.05)
    plt.tight_layout(); plt.show()


## 5. Depth montage: what the z slab looks like

The encoder receives a `(1, z_thickness, H, W)` tensor per crop. This montages the z slices of one global so you can see what's stacked together.

In [ ]:
volume = g_t1[0, 0]  # first global at t1
montage = z_montage(volume)
mlims = percentile_limits(volume, 1.0, 99.0)
fig, ax = plt.subplots(figsize=(12, 12 * (montage.shape[0] / montage.shape[1])))
show_gray(ax, montage, f'global 0 t1 z-stack montage (z_thickness={cfg.z_thickness})', mlims)
plt.tight_layout(); plt.show()


## 6. 16 independent augmentations of the same crop

Same anchor every time; only the augmentation roll changes. Use this to gauge whether augmentations are strong enough to corrupt surface noise but not so strong they destroy biology. The anchor is a freshly-loaded global crop at the current sample's `(t1, z, y, x)`.

In [ ]:
import torch
from data.augmentations import RandomIntensity, RandomGamma, GaussianNoise, GaussianBlur, PerCropNormalize

# short label per aug type for the panel titles.
_AUG_LABEL = {
    RandomGamma: 'gamma',
    RandomIntensity: 'intensity',
    GaussianNoise: 'noise',
    GaussianBlur: 'blur',
    PerCropNormalize: 'norm',
}


def augment_with_log(pipeline, x):
    """step through the pipeline manually so we can record which augs fired.

    mirrors `_Aug.__call__`: skip if disabled, roll `torch.rand() < p`, else apply.
    PerCropNormalize has `p=1.0` and always fires when enabled.
    """
    fired = []
    for t in pipeline.augs:
        if not t.cfg.enabled:
            continue
        if torch.rand(()).item() >= t.cfg.p:
            continue
        x = t._apply(x)
        fired.append(_AUG_LABEL.get(type(t), type(t).__name__))
    return x, fired


gy0 = int(meta['global_y'][0])
gx0 = int(meta['global_x'][0])
base = load_crop_tensor(src, int(meta['t1']), int(meta['z']),
                        gy0, gx0, cfg.global_crop_size)
augment = build_augmentations(cfg)

samples = []
labels = []
for _ in range(16):
    out, fired = augment_with_log(augment, base.clone())
    samples.append(mean_projection(out[0].numpy()))
    labels.append(', '.join(fired) if fired else 'none')
lims = percentile_limits(np.concatenate([s.ravel() for s in samples]), 1.0, 99.0)

fig, axes = plt.subplots(4, 4, figsize=(11, 11))
for ax, img, label in zip(axes.flat, samples, labels):
    show_gray(ax, img, label, lims)
plt.suptitle('16 augmented variants of the same global anchor (mean z)', y=1.005)
plt.tight_layout(); plt.show()


## 7. Each augmentation in isolation

Same crop, but only one augmentation type is active at a time (and forced to `p=1.0`). Per-crop normalization is always applied last so panels share a comparable scale.

In [ ]:
def isolate(name):
    c = copy.deepcopy(cfg)
    for attr in ('aug_intensity', 'aug_gamma', 'aug_noise', 'aug_blur'):
        getattr(c, attr).enabled = (attr == name)
        if attr == name:
            getattr(c, attr).p = 1.0
    c.aug_percrop_norm.enabled = True
    return build_augmentations(c)

def norm_only():
    c = copy.deepcopy(cfg)
    for attr in ('aug_intensity', 'aug_gamma', 'aug_noise', 'aug_blur'):
        getattr(c, attr).enabled = False
    return build_augmentations(c)

modes = [
    ('norm only',  norm_only()),
    ('intensity',  isolate('aug_intensity')),
    ('gamma',      isolate('aug_gamma')),
    ('noise',      isolate('aug_noise')),
    ('blur',       isolate('aug_blur')),
    ('full stack', build_augmentations(cfg)),
]
imgs = [(name, mean_projection(aug(base.clone())[0].numpy())) for name, aug in modes]
lims = percentile_limits(np.concatenate([img.ravel() for _, img in imgs]), 1.0, 99.0)

fig, axes = plt.subplots(1, len(modes), figsize=(3 * len(modes), 3.5))
for ax, (name, img) in zip(axes, imgs):
    show_gray(ax, img, name, lims)
plt.suptitle('per-augmentation isolation (mean z)', y=1.02)
plt.tight_layout(); plt.show()


## 8. A full batch as the DataLoader hands it to the model

This pulls the first batch the DataLoader would emit. Per sample we show one row for globals_t1, one for globals_t2, and (if present) one for locals, all in normalized units, post-augmentation. The `weak_label` column shows the `(dataset_idx, frame_gap)` long tensor that the loss will consume.

In [ ]:
from torch.utils.data import DataLoader

batch_size = 2
loader = DataLoader(
    ds,
    batch_size=batch_size,
    num_workers=0,  # in-process so the notebook stays simple
    worker_init_fn=worker_init_fn,
)

batch = next(iter(loader))
g_t1_b = batch['globals_t1'].numpy()  # (B, n_global, 1, z, h, w)
g_t2_b = batch['globals_t2'].numpy()
loc_b = batch['locals'].numpy() if 'locals' in batch else None
weak_b = batch['weak_label']
metas = batch['meta']

print('globals_t1 batch shape:', g_t1_b.shape)
print('globals_t2 batch shape:', g_t2_b.shape)
if loc_b is not None:
    print('locals batch shape    :', loc_b.shape)
print('weak_label batch shape:', tuple(weak_b.shape))
print('weak_label[:, 0] (dataset_idx):', weak_b[:, 0].tolist())
print('weak_label[:, 1] (frame_gap)  :', weak_b[:, 1].tolist())

# shared display limits across both streams.
pool = np.concatenate([g_t1_b.ravel(), g_t2_b.ravel()])
glims = percentile_limits(pool, 1.0, 99.0)
llims = percentile_limits(loc_b, 1.0, 99.0) if loc_b is not None else None

n_g_b = g_t1_b.shape[1]
n_l_b = loc_b.shape[1] if loc_b is not None else 0
ncols = max(n_g_b, n_l_b, 1)
rows_per_sample = 3 if loc_b is not None else 2
total_rows = batch_size * rows_per_sample

fig, axes = plt.subplots(total_rows, ncols, figsize=(2.4 * ncols, 2.6 * total_rows), squeeze=False)
for b in range(batch_size):
    t1 = int(metas['t1'][b]); t2 = int(metas['t2'][b])
    ds_idx = int(metas['dataset_idx'][b])
    fg = int(metas['frame_gap'][b])
    tag = f'b{b} ds={ds_idx} gap={fg}'
    row_g1 = b * rows_per_sample
    row_g2 = row_g1 + 1
    for i in range(ncols):
        if i < n_g_b:
            show_gray(axes[row_g1, i], mean_projection(g_t1_b[b, i, 0]),
                      f'{tag} g{i} t1={t1}', glims)
            show_gray(axes[row_g2, i], mean_projection(g_t2_b[b, i, 0]),
                      f'{tag} g{i} t2={t2}', glims)
        else:
            axes[row_g1, i].axis('off')
            axes[row_g2, i].axis('off')
    if loc_b is not None:
        row_l = row_g2 + 1
        parents = metas['local_parent'][b].tolist()
        for i in range(ncols):
            if i < n_l_b:
                show_gray(axes[row_l, i], mean_projection(loc_b[b, i, 0]),
                          f'{tag} loc{i} p={parents[i]}', llims)
            else:
                axes[row_l, i].axis('off')

plt.suptitle(f'batch_size={batch_size} as the model receives it', y=1.005)
plt.tight_layout(); plt.show()
